# Day 3 - practical task

### 0. Importing the necessary libraries

In [9]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
    )
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    recall_score,
    precision_score,
    confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
import numpy as np
import pandas as pd

### 1. Loading the breast cancer classification dataset

In [10]:
data = load_breast_cancer()

X = data.data
# How well it finds the bad cancer (malignant)
y = (data.target == 0).astype(int)

# Cheking the dataset
print("Missing values:", np.isnan(X).sum())
print("Infinite values:", np.isinf(X).sum())

# Checking the target's class distribution
print(pd.Series(y).value_counts())
print(pd.Series(y).value_counts(normalize=True))

Missing values: 0
Infinite values: 0
0    357
1    212
Name: count, dtype: int64
0    0.627417
1    0.372583
Name: proportion, dtype: float64


### 2. Split into training and test sets

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

### 3. Define models

In [12]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Boosting": GradientBoostingClassifier(random_state=42),
}

### 4. Tune-in hyperparameters

In [13]:
# Random Forest parameters
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

# Decision Tree parameters
dt_params = {
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

# Boosting parameters
bst_params = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
}

grid_searches = {
    "Random Forest": GridSearchCV(
    estimator=models["Random Forest"],
    param_grid=rf_params,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='recall'
),
    "Decision Tree": GridSearchCV(
    estimator=models["Decision Tree"],
    param_grid=dt_params,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='recall'
),
    "Gradient Boosting": GridSearchCV(
    estimator=models["Boosting"],
    param_grid=bst_params,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='recall'
),
}

for name, search in grid_searches.items():
    search.fit(X_train, y_train)
    print(f"{name}")
    print("Best Parameters:", search.best_params_)
    print("Best CV recall:", search.best_score_, "\n")

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Random Forest
Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best CV recall: 0.9470588235294117 

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Decision Tree
Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV recall: 0.9117647058823529 

Fitting 5 folds for each of 64 candidates, totalling 320 fits
Gradient Boosting
Best Parameters: {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Best CV recall: 0.9529411764705882 



In [14]:
best_rf = grid_searches["Random Forest"].best_estimator_
best_dt = grid_searches["Decision Tree"].best_estimator_
best_bst = grid_searches["Gradient Boosting"].best_estimator_

comparison_models = {
    "Decision Tree (Default)": models["Decision Tree"],
    "Decision Tree (Tuned)": best_dt,
    "Random Forest (Default)": models["Random Forest"],
    "Random Forest (Tuned)": best_rf,
    "Gradient Boosting (Default)": models["Boosting"],
    "Gradient Boosting (Tuned)": best_bst,
}

### 5. Train

In [15]:
results = []
for name, model in comparison_models.items():
    
    if "Default" in name:
        model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_score = roc_auc_score(y_test, probabilities)

    cm = confusion_matrix(y_test, predictions, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    train_score = model.score(X_train, y_train)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 score": f1,
        "ROC AUC": roc_score,
        "Train accuracy": train_score,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    })

### 6. Compare results

In [16]:
results_df = pd.DataFrame(results)

display(
    results_df.style.format({
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1 score": "{:.3f}",
        "ROC AUC": "{:.3f}",
        "Train accuracy": "{:.3f}",
        "Test accuracy": "{:.3f}",
    })
)

,Model,Accuracy,Precision,Recall,F1 score,ROC AUC,Train accuracy,TN,FP,FN,TP
0,Decision Tree (Default),0.930,0.905,0.905,0.905,0.925,1.000,68,4,4,38
1,Decision Tree (Tuned),0.930,0.905,0.905,0.905,0.925,1.000,68,4,4,38
2,Random Forest (Default),0.974,1.000,0.929,0.963,0.993,1.000,72,0,3,39
3,Random Forest (Tuned),0.974,1.000,0.929,0.963,0.993,1.000,72,0,3,39
4,Gradient Boosting (Default),0.965,1.000,0.905,0.950,0.995,1.000,72,0,4,38
5,Gradient Boosting (Tuned),0.947,0.974,0.881,0.925,0.995,1.000,71,1,5,37


### 7. Conclusion
Given the results, *Decision Tree* and *Random Forest* choose the best hyperparameters which are the default ones. *Gradient boost* chose different hyperparameters meaning that tuning slightly degraded the performance. All models `Train accuracy` was 1 while test `Accuracy` was lower suggesting **overfitting**. Overall *Random Forest* performed the best because it had the best `Accuracy` (0.974), lowest FN (3), most TP (39), best Recall (0.929) and F1 scores(0.963). So, Hyperparameter tuning doesn't guarantee better performance.